# NB3 · Model kurulumu ve değerlendirme

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---

Bu defterde model kurmak on beş satır sürer. Defterin geri kalanı o modelin ne kadar işe
yaradığını ölçmeye ayrılmıştır.

Kontrol hücrelerinin sayısı NB2'ye göre azdır; buradaki kontroller yalnızca sessiz hataları
hedefler.


## Hazırlık


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for modul in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
              'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{modul}', modul)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'tr'


In [ ]:
# Sabit hücre. NB2'de kurduğunuz kohortu yeniden oluşturur.
import pipeline as pl

durum = pl.prepare(verbose=False)
kohort = durum['cohort']
ozellikler = durum['features']
kohort = kohort.rename(columns={'prolonged_stay': 'hedef'})
print(f'Kohort hazır: {len(kohort)} yatış, {len(ozellikler)} öznitelik.')


---

## Adım 1 · Verinin ayrılması

Kohortta bir hastanın birden fazla yoğun bakım yatışı bulunabilir. Veri satır düzeyinde
rastgele ayrılırsa aynı hastanın bir yatışı eğitim, diğeri test kümesine düşer. Model o
hastayı tanıdığı için test başarımı yükselir ve bu yükselme gerçek değildir.

Üretken yapay zekâ araçlarının en sık yaptığı sessiz hatalardan biridir. Ayrım istendiğinde
araç varsayılan olarak satırları rastgele böler; hasta kimliğini dikkate alması ayrıca
istenmelidir.


### İstem 1

```
kohort adında bir pandas DataFrame var. Her satır bir yoğun bakım yatışı. subject_id
sütunu hasta kimliğini, hedef sütunu ikili sonucu veriyor. Bir hastanın birden fazla
yatışı olabilir.

Veriyi eğitim ve test olarak ayıran tek bir Python hücresi yaz. Ayrımı hasta düzeyinde
yap: Aynı hasta iki kümede birden bulunmasın. Test oranı 0.30 olsun.

KABUL ÖLÇÜTLERİ
egitim ve test adında iki DataFrame üret.
İki kümede ortak subject_id bulunmasın.
Her iki kümede de hedef sütunu hem 0 hem 1 değerini alsın.
İki kümenin satır sayısını ve olay oranını ekrana yaz.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 1


In [ ]:
checks.check_split(egitim, test, target='hedef', patient_id='subject_id')


---

## Adım 2 · Modelin kurulması

Ölçekleyici, tamamlayıcı ve kodlayıcı yalnızca eğitim kümesi üzerinde öğrenilmeli, test
kümesine uygulanmalıdır. Bu adımlar ayrımdan önce tüm veri üzerinde çalıştırıldığında
test kümesindeki değerler eğitim sürecine taşınır.

Söz konusu hata hiçbir uyarı üretmez, başarımı yükseltir ve kod okunduğunda adım sırası
mantıklı görünür. Bütün ön işleme zincirini tek bir Pipeline içine almak, hatayı yapısal
olarak imkânsız kılar.


### İstem 2

```
egitim ve test DataFrame'lerini kullanarak temel bir model kur. Tek bir Python hücresi
yaz.

Sayısal sütunlarda ortanca ile tamamlama ve ölçekleme, kategorik sütunlarda en sık
değerle tamamlama ve one-hot kodlama uygula. Sınıflandırıcı olarak lojistik regresyon
kullan ve sınıf ağırlıklarını dengele.

subject_id, hadm_id, stay_id, intime ve hedef sütunlarını öznitelik olarak kullanma.

Hiperparametre araması yapma. Temel model iyi olmak için değil, aşılmak için kurulur.

KABUL ÖLÇÜTLERİ
model adında eğitilmiş bir scikit-learn Pipeline üret.
Bütün ön işleme adımları bu Pipeline içinde olsun; Pipeline dışında hiçbir fit
çağrısı bulunmasın.
ozellik_listesi adında, kullanılan sütun adlarının listesini üret.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 2


In [ ]:
checks.check_model(model)


---

## Adım 3 · Tahminlerin üretilmesi

Model sınıf etiketi değil olasılık üretmelidir. Eşik, olasılık üzerinden ve klinik
gerekçeyle belirlenir. Doğrudan sınıf etiketi üreten bir model, eşiği kendi varsayılanına
sabitlemiş demektir ve o varsayılan klinik bir karar değildir.


### İstem 3

```
Eğitilmiş model ile test kümesi üzerinde tahmin üret. Tek bir Python hücresi yaz.

KABUL ÖLÇÜTLERİ
olasilik adında bir dizi üret. Test kümesindeki her satır için pozitif sınıf
olasılığını içersin.
Uzunluğu test kümesinin satır sayısına eşit olsun.
Değerler 0 ile 1 arasında bulunsun.
```


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 3


In [ ]:
checks.check_predictions(olasilik, n_expected=len(test))


---

## Doğruluk tuzağı

Aşağıdaki iki hücre sabittir. Birincisini çalıştırıp rakamı okuyunuz, ardından
ikinciyi çalıştırınız.


In [ ]:
dogruluk = (( olasilik >= 0.5).astype(int) == test['hedef'].values).mean()
print(f'Test doğruluğu: {dogruluk:.1%}')


In [ ]:
bos = ev.null_comparison(test['hedef'])
print(f"Hiçbir şey öğrenmeyen kural: {bos['accuracy']:.1%}")
print()
print('İki değer arasındaki fark, modelin gerçekten kattığı değerdir.')


Dengesiz bir klinik problemde doğruluk, modelin ne yaptığını değil hastalığın ne kadar
nadir olduğunu ölçer. Yüzde yedi prevalansta her zaman olumsuz sınıfı söyleyen bir kural
yüzde 93 doğruluk verir.


---

## Değerlendirme raporu

Aşağıdaki hücre defterin sabit değerlendirme bölümüdür ve altı başlığı sırayla uygular:
Güven aralığıyla ayrım gücü, kalibrasyon, çalışma noktası, klinik karşılık, alt grup
dökümü ve boş karşılaştırma.

`target_sensitivity` değeri problem kartınızdaki maliyet dengesinin karşılığıdır.
Kaçırma daha pahalıysa yüksek tutulur. Bu bir teknik varsayılan değil, klinik bir
karardır.


In [ ]:
rapor = ev.honest_report(
    test['hedef'], olasilik,
    groups=test['gender'] if 'gender' in test.columns else None,
    target_sensitivity=0.80,
    label='temel model · yoğun bakımda uzamış kalış',
)


In [ ]:
sekil = ev.plot_curves(test['hedef'], olasilik)


## Raporun okunması

Güven aralığı 0,5 değerini içeriyorsa model şanstan ayırt edilemiyor demektir; nokta
tahmini ne olursa olsun bu böyledir.

Kalibrasyon eğimi birin altındaysa model aşırı güvenlidir. Eşik belirleyen bir klinisyen
için bu, eşiğin sandığından farklı bir yerde durması anlamına gelir.

Klinik karşılık satırı, her yüz hastada kaç uyarının çıktığını ve kaçının doğru olduğunu
verir. Bu satır dersteki Epic Sepsis Model örneğinin buradaki karşılığıdır.

Alt grup dökümünde bazı gruplar için sayı yerine yetersiz örneklem ifadesi yer alır. Bu
bir eksiklik değil bulgudur: Hiç test edilmemiş bir grup için sistemin adil olduğu
gösterilemez.

Bu senaryoda raporun olumsuz çıkması beklenmektedir. Yüz hastalık bir kümede güven
aralığı geniştir ve modelin boş karşılaştırmaya üstünlüğü küçüktür. Kendi kurduğunuz
sistem hakkında bu sonuca varmak, bir başkasının başarısızlığını dinlemekten farklıdır.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
